# 06 – Greedy Inference and Metrics

Disaggregate mains power using greedy HMM approach and compute evaluation metrics.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from src.config import FULL_HOUSES, TARGET_APPLIANCES, PROCESSED_DATA_DIR, MODELS_DIR, INFERENCE_ORDER
from src.hmm_train import load_model
from src.inference_greedy import greedy_disaggregate
from src.metrics import evaluate_all
from src.plots import plot_comparison

In [ ]:
test_house = FULL_HOUSES[0]
data_path = Path('..') / PROCESSED_DATA_DIR / f'house_{test_house}.parquet'
model_dir = Path('..') / MODELS_DIR / f'test_house_{test_house:02d}'

if not data_path.exists() or not model_dir.exists():
    print('Run preprocess_all.py and run_loho_train.py first.')
else:
    df = pd.read_parquet(data_path)
    models = {}
    for app in TARGET_APPLIANCES:
        mp = model_dir / f'{app}.pkl'
        if mp.exists():
            models[app] = load_model(str(mp))
    preds = greedy_disaggregate(df['mains'], models, order=INFERENCE_ORDER)
    apps_to_eval = [a for a in TARGET_APPLIANCES if a in df.columns and a in preds.columns]
    metrics = evaluate_all(df[apps_to_eval], preds[apps_to_eval], house_id=test_house)
    print(metrics.to_string(index=False))

In [ ]:
if 'preds' in dir():
    for app in apps_to_eval[:2]:
        ax = plot_comparison(df[app].iloc[:2000], preds[app].iloc[:2000], appliance=app)
        plt.tight_layout()
        plt.show()